# Kitaev chain 200 site DMRG

Created 03/06/2025

Objectives:
* Following [the original Kitaev chain paper](https://dx.doi.org/10.1070/1063-7869/44/10S/S29), let $$H = \sum_j \left(-t (a_j^\dagger a_{j+1} + a^\dagger_{j+1} a_j - a_j a_{j+1} - a^\dagger{j+1} a^\dagger_j) - (1-t)(a_j^\dagger a_j -1/2)\right)$$.

# Imports 

In [1]:
from itertools import product

In [2]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [3]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS

In [5]:
from tenpy.networks.mpo import MPO

from tenpy.networks.site import FermionSite
from tenpy.models.lattice import Chain
from tenpy.models.model import CouplingMPOModel

# Define model

In [29]:
class KitaevChain(CouplingMPOModel):
    default_lattice = "Chain"
    force_default_lattice = True

    def init_sites(self, model_params):
        return FermionSite(conserve=None)

    def init_terms(self, model_params):
        # Read off model parameters
        t = model_params.get('interpolation', 0)

        self.add_coupling(-t, 0, 'Cd', 0, 'C', 1, plus_hc=True)
        self.add_coupling(t, 0, 'C', 0, 'C', 1, plus_hc=True)


        self.add_onsite(-(1-t), 0, 'N', plus_hc=False)
        self.add_onsite((1-t)*0.5, 0, 'Id', plus_hc=False)

# Run DMRG

In [8]:
dmrg_params = {
    "trunc_params": {"chi_max": 16, "chi_min": 1, "svd_min": 1.e-8},
    "min_sweeps":10,
    "max_sweeps":200,
    "mixer": True,
    "combine":False,
    'decay':2,
    'amplitude':10e-1,
    'disable_after':60,
    'update_env':0
}

In [9]:
import h5py
from tenpy.tools import hdf5_io

In [10]:
interpolation_values = np.linspace(0, 1 , 21)

In [11]:
interpolation_values

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95, 1.  ])

In [34]:
num_sites = 100

for t in interpolation_values:
        print(f"Commencing interpolation={t} model")
        model=KitaevChain({'interpolation': t, 'L': num_sites})
        
        psi = MPS.from_lat_product_state(model.lat, [['empty',],]*num_unit_cells)
        psi.canonical_form()
    
        eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
        e, psi = eng.run()

        print(f"Energy: {e}\n")

        data = {
            "wavefunction": psi,
            "energy": e,
            "paramters": {"interpolation": t}
        }

        t_string = str(int(100*t))
        one_minus_t_string = str(int(100*(1-t)))

        file_name = f'{t_string}_pc_triv_{one_minus_t_string}_pc_non_triv'
    
        filename = (
            r"../data/kitaev_chain_200_site_dmrg/{}"
            .format(file_name)
        )

        filename += ".h5"

        with h5py.File(filename, 'w') as f:

            hdf5_io.save_to_hdf5(f, data)

Commencing interpolation=0.0 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -50.0

Commencing interpolation=0.05 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -47.630351786228566

Commencing interpolation=0.1 model


...

Energy: -89.17084727606158

Commencing interpolation=0.9500000000000001 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -94.0667770562812

Commencing interpolation=1.0 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['amplitude', 'decay', 'disable_after', 'update_env']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -99.00000000000006

